In [ ]:
# Summary: 
# Automate scraping pages of battles from PS, getting their respective replays, 
# and saving them to disk. The 'main' function is toward the end.

In [1]:
# Some basic imports
import numpy as np
import pandas as pd

import json, os, time
import logging, requests
import itertools
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
from urllib.parse import urlencode, urljoin

import re # for regex things

# Useful 'constants'
SEARCH_BASE_URL = "https://replay.pokemonshowdown.com/api/replays/search"
REPLAY_BASE_URL = "https://replay.pokemonshowdown.com/"
USER_BASE_URL = "https://pokemonshowdown.com/users/"

REPLAY_DIR = Path('../data/replays/')

# Only needed if you want messages to appear when an error, etc. occurs
logger = logging.getLogger()
logger.setLevel(logging.ERROR)

# Getting pages/replays
-----------

In [2]:
# Dataclasses
# ===============================================

# Roughly, we 
#    1. Pull pages full of Battles;
#    2. Get `battle_id`s from each Battle;
#    3. Pull and write the Replay using the `battle_id`.
# 
# The `Battle` dataclass is unnecessary at present, but we can use it later if needed.

@dataclass
class Battle:
    uploadtime: int
    id: str
    format: str
    players: list[str]
    rating: Optional[int] = None

@dataclass
class Replay:
    id: str
    format: str
    players: list[str]
    log: str
    inputlog: str
    uploadtime: int
    views: int
    formatid: str
    rating: object = None
    private: int = 0
    password: object = None

In [3]:
# ensuring directory exists
# ===============================================
def ensure_dir(name):
    try:
        Path(name).mkdir(exist_ok=True)
    except OSError as e:
        logger.error("failed to create dir %s: %s", name, e)
        raise SystemExit(1)

In [4]:
# %%%%%%%%% 'Getters' %%%%%%%%%%%%%%%%%%%%%%%%%%%
def get_page(page_num, fmt='') -> list[dict]:
    '''
    Returns a list of dicts, with each dict corresponding to a `json` of a battle.
    The dicts have keys/values like that of the Battle dataclass; `id` is the only crucial one.

    Leaving `fmt=''` searches for battles of any format.
    '''
    
    params = urlencode({"format": fmt, "page": str(page_num)}) 
    url = f"{SEARCH_BASE_URL}?{params}"

    # This deals with an apparent quirk in PS's search: the first-page results are fine, 
    # but subsequent searches seem to want this `Referer` in the GET request.
    headers = {}
    if page_num > 1: 
        headers = {"Referer": f"https://replay.pokemonshowdown.com/?format={fmt}&page={page_num}"}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error("failed to get page %d: %s", page_num, e)
        return []

    # The API response starts with a leading character ']' before valid JSON,
    # so remove it
    raw = response.content[1:]
    
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        logger.error("failed to parse page %d: %s", page_num, e)
        return []
    return data


# ===========================
def get_replay(battle_id: str) -> Replay:
    '''
    Returns a Replay object pulled using the unique battle_id.
    
    Much like `get_page` in function .
    '''
    
    url = REPLAY_BASE_URL + f'{battle_id}' + '.json'
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
    except requests.RequestException as e:
        logger.error("failed to get replay: %s", e)
        return None

    try:
        data = response.json()
    except json.JSONDecodeError as e:
        logger.error("failed to parse replay: %s", e)
        return None

    return Replay(
        id=data.get("id", ""),
        format=data.get("format", ""),
        players=data.get("players", []),
        log=data.get("log", ""),
        inputlog=data.get("inputlog", ""),
        uploadtime=data.get("uploadtime", 0),
        views=data.get("views", 0),
        formatid=data.get("formatid", ""),
        rating=data.get("rating"),
        private=data.get("private", 0),
        password=data.get("password"),
    )

# The 'main' scraper

In [5]:
# Definition
def scrape_replays(page_num, fmt=""):
    '''
    Search page, get battle Replays, and write to file.
    '''
    
    # main page-scraping
    page = get_page(page_num, fmt)
    
    if page == [] : 
        print(f"Failed to get page {page_num}.")
        return None
    
    for battle in page :
        # main replay-scraping
        replay = get_replay(battle['id'])
        if replay is None:
            logger.error("failed to get replay %s", battle['id'])
            return None
        
        # Writing
        if fmt == "" : 
            out_path = REPLAY_DIR / f"{replay.id}.json"
        else : 
            out_path = REPLAY_DIR / f"2_{fmt}/{replay.id}.json"
        
        try:
            out_path.write_text(json.dumps(replay.__dict__), encoding="utf-8")
        except OSError as e:
            logger.error("failed to save replay %s : %s", battle['id'], e)

In [10]:
REPLAY_DIR = Path('../data/replays/')
fmt = 'gen9-randombattle'

error_ids = ['gen9randombattle-2641745504',
 'gen9randombattle-2641744694',
 'gen9randombattle-2641746190',
 'gen9randombattle-2641744528',
 'gen9randombattle-2641745810',
 'gen9randombattle-2641745847',
 'gen9randombattle-2641747396',
 'gen9randombattle-2641745523',
 'gen9randombattle-2641745221',
 'gen9randombattle-2641746967',
 'gen9randombattle-2641747273',
 'gen9randombattle-2641745052',
 'gen9randombattle-2641541263',
 'gen9randombattle-2641746624',
 'gen9randombattle-2641743200',
 'gen9randombattle-2641745696',
 'gen9randombattle-2641747042']


for btid in error_ids :
    # main replay-scraping
    replay = get_replay(btid)
    if replay is None:
        logger.error("failed to get replay %s", btid)
    
    # Writing
    if fmt == "" : 
        out_path = REPLAY_DIR / f"{replay.id}.json"
    else : 
        out_path = REPLAY_DIR / f"2_{fmt}/{replay.id}.json"
    
    try:
        out_path.write_text(json.dumps(replay.__dict__), encoding="utf-8")
    except OSError as e:
        logger.error("failed to save replay %s : %s", btid, e)

In [8]:
# Automation/Running

FORMAT='gen9-randombattle';

ensure_dir('../data/replays/')
ensure_dir('../data/replays/2_'+FORMAT)

PAUSE = 3; # seconds between pages; NOTE: Should be at least 1 to safely avoid connection denial.

# Note: even trying searching manually, it seems results stop at page 100
for j in range(1,100): 
    print(f"Now working on page {j}.")
    
    scrape_replays(j, fmt=FORMAT)
    
    print(f"\tDone; taking a {PAUSE} second break.")
    time.sleep(PAUSE)

Now working on page 1.
	Done; taking a 3 second break.
Now working on page 2.
	Done; taking a 3 second break.
Now working on page 3.
	Done; taking a 3 second break.
Now working on page 4.
	Done; taking a 3 second break.
Now working on page 5.
	Done; taking a 3 second break.
Now working on page 6.
	Done; taking a 3 second break.
Now working on page 7.
	Done; taking a 3 second break.
Now working on page 8.
	Done; taking a 3 second break.
Now working on page 9.
	Done; taking a 3 second break.
Now working on page 10.
	Done; taking a 3 second break.
Now working on page 11.
	Done; taking a 3 second break.
Now working on page 12.
	Done; taking a 3 second break.
Now working on page 13.
	Done; taking a 3 second break.
Now working on page 14.
	Done; taking a 3 second break.
Now working on page 15.


failed to get replay: HTTPSConnectionPool(host='replay.pokemonshowdown.com', port=443): Read timed out. (read timeout=30)
failed to get replay gen9randombattle-2641661141


	Done; taking a 3 second break.
Now working on page 16.
	Done; taking a 3 second break.
Now working on page 17.
	Done; taking a 3 second break.
Now working on page 18.
	Done; taking a 3 second break.
Now working on page 19.
	Done; taking a 3 second break.
Now working on page 20.
	Done; taking a 3 second break.
Now working on page 21.
	Done; taking a 3 second break.
Now working on page 22.
	Done; taking a 3 second break.
Now working on page 23.
	Done; taking a 3 second break.
Now working on page 24.
	Done; taking a 3 second break.
Now working on page 25.
	Done; taking a 3 second break.
Now working on page 26.
	Done; taking a 3 second break.
Now working on page 27.
	Done; taking a 3 second break.
Now working on page 28.
	Done; taking a 3 second break.
Now working on page 29.


failed to get page 29: HTTPSConnectionPool(host='replay.pokemonshowdown.com', port=443): Read timed out. (read timeout=10)


Failed to get page 29.
	Done; taking a 3 second break.
Now working on page 30.
	Done; taking a 3 second break.
Now working on page 31.


...

	Done; taking a 3 second break.
Now working on page 32.
	Done; taking a 3 second break.
Now working on page 33.
	Done; taking a 3 second break.
Now working on page 34.
	Done; taking a 3 second break.
Now working on page 35.
	Done; taking a 3 second break.
Now working on page 36.
	Done; taking a 3 second break.
Now working on page 37.
	Done; taking a 3 second break.
Now working on page 38.
	Done; taking a 3 second break.
Now working on page 39.
	Done; taking a 3 second break.
Now working on page 40.
	Done; taking a 3 second break.
Now working on page 41.
	Done; taking a 3 second break.
Now working on page 42.
	Done; taking a 3 second break.
Now working on page 43.
	Done; taking a 3 second break.
Now working on page 44.
	Done; taking a 3 second break.
Now working on page 45.
	Done; taking a 3 second break.
Now working on page 46.
	Done; taking a 3 second break.
Now working on page 47.
	Done; taking a 3 second break.
Now working on page 48.
	Done; taking a 3 second break.
Now working on p

failed to get replay: HTTPSConnectionPool(host='replay.pokemonshowdown.com', port=443): Read timed out. (read timeout=30)
failed to get replay gen9randombattle-2641438520


	Done; taking a 3 second break.
Now working on page 64.
	Done; taking a 3 second break.
Now working on page 65.
	Done; taking a 3 second break.
Now working on page 66.
	Done; taking a 3 second break.
Now working on page 67.
	Done; taking a 3 second break.
Now working on page 68.
	Done; taking a 3 second break.
Now working on page 69.
	Done; taking a 3 second break.
Now working on page 70.
	Done; taking a 3 second break.
Now working on page 71.
	Done; taking a 3 second break.
Now working on page 72.
	Done; taking a 3 second break.
Now working on page 73.
	Done; taking a 3 second break.
Now working on page 74.
	Done; taking a 3 second break.
Now working on page 75.
	Done; taking a 3 second break.
Now working on page 76.
	Done; taking a 3 second break.
Now working on page 77.
	Done; taking a 3 second break.
Now working on page 78.
	Done; taking a 3 second break.
Now working on page 79.
	Done; taking a 3 second break.
Now working on page 80.
	Done; taking a 3 second break.
Now working on p

failed to get replay: HTTPSConnectionPool(host='replay.pokemonshowdown.com', port=443): Read timed out. (read timeout=30)
failed to get replay gen9randombattle-2641089712


	Done; taking a 3 second break.
Now working on page 95.
	Done; taking a 3 second break.
Now working on page 96.
	Done; taking a 3 second break.
Now working on page 97.
	Done; taking a 3 second break.
Now working on page 98.
	Done; taking a 3 second break.
Now working on page 99.
	Done; taking a 3 second break.


In [7]:
REPLAY_DIR = Path('./replays/gen9-randombattle')

battleid_list = []

for item in REPLAY_DIR.iterdir():
    if (item.is_file() and item.name[0]!='.'): # skipping hidden files
        try:
            with open(REPLAY_DIR/f'{item.name}', 'r') as file:
                x = json.load(file)
                battleid_list.append(x['id'])
        except UnicodeDecodeError as e:
            print(f"{item.name}")
    else : 
        print(f'{item.name}')
        continue

.DS_Store
.ipynb_checkpoints


In [ ]:
PAUSE = 1; # seconds between pulls
REPLAY_DIR = Path('./replays/gen9-randombattle')
# Note: even trying searching manually, it seems results stop at page 100
for j in range(4000,len(battleid_list)): 
    
    if j % 20 == 0 : print(f"Now working on replay {j}.")
    # main replay-scraping
    replay = get_replay(battleid_list[j])
    
    if replay is None:
        logger.error("failed to get replay %s", battleid_list[j])
        continue
    
    # Writing
    out_path = REPLAY_DIR / f"{replay.id}.json"
    
    try:
        out_path.write_text(json.dumps(replay.__dict__), encoding="utf-8")
    except OSError as e:
        logger.error("failed to save replay %s : %s", battleid_list[j], e)
    time.sleep(PAUSE)